<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/notebooks/06_FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 6: First Fine-Tuning Task (Resume Classification)

**Goal:**
- Fine-tune `microsoft/deberta-v3-base` on our Resume text to classify Job Categories.
- Use the exact same Stratified Splits from Phase 2/5 to ensure scientific comparability with the TF-IDF Baseline.
- Output detailed evaluation metrics (Macro F1) to quantify the Deep Learning performance lift.

In [ ]:
# Install Hugging Face ecosystems
!pip install transformers datasets evaluate accelerate scikit-learn seaborn matplotlib sentencepiece

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
from sklearn.metrics import confusion_matrix, classification_report

# Ensure GPU is used if available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# 1. Load Data & Ensure Same Split as Baseline
splits_df = pd.read_csv("resume_metadata_split.csv")
nlp_df = pd.read_csv("nlp_processed_resumes.csv")

df = pd.merge(nlp_df, splits_df[['filename', 'split']], on='filename', how='inner')

# Create Label Encodings
labels = sorted(df['category'].unique())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)

df['label'] = df['category'].map(label2id)

# Convert to Hugging Face Dataset
hg_dataset = DatasetDict({
    'train': Dataset.from_pandas(df[df['split'] == 'train'][['raw_clean_text', 'label']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(df[df['split'] == 'val'][['raw_clean_text', 'label']].reset_index(drop=True)),
    'test': Dataset.from_pandas(df[df['split'] == 'test'][['raw_clean_text', 'label']].reset_index(drop=True))
})
print(hg_dataset)

In [ ]:
# 2. Tokenization
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Standard transformers max_length is 512. We truncate long resumes.
    return tokenizer(examples["raw_clean_text"], padding="max_length", truncation=True, max_length=512)

print("Tokenizing datasets...")
tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)
# Remove the raw text column so PyTorch can handle the tensors
tokenized_datasets = tokenized_datasets.remove_columns(["raw_clean_text"])

In [ ]:
# 3. Metrics Setup (Macro F1 is critical for imbalanced classes)
metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")
metric_prec = evaluate.load("precision")
metric_rec = evaluate.load("recall")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    # average='macro' is extremely important here
    precision = metric_prec.compute(predictions=predictions, references=labels, average="macro", zero_division=0)["precision"]
    recall = metric_rec.compute(predictions=predictions, references=labels, average="macro", zero_division=0)["recall"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    
    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    }

In [ ]:
# 4. Initialize Model and Trainer
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# We use focal loss or class weights natively by dealing with batch sizes, 
# but for simplicity in this baseline fine-tuning, standard CrossEntropy is used.
training_args = TrainingArguments(
    output_dir="./deberta-resume-classifier",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8, # T4 GPU limit for DeBERTa
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True, # Enable mixed precision for speed
    logging_dir='./logs',
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
# 5. Train the Model
print("Starting Fine-Tuning Process...")
trainer.train()

In [ ]:
# 6. Final Evaluation on UNSEEN TEST SET
print("\n--- Final Evaluation on Test Set ---")
predictions, labels, metrics = trainer.predict(tokenized_datasets["test"])

print(f"Test Accuracy: {metrics['test_accuracy']:.4f}")
print(f"Test Macro F1: {metrics['test_macro_f1']:.4f}")
print("Compare this Macro F1 against your Phase 5 Baseline SVM!")

# Confusion Matrix Plotting
pred_labels = np.argmax(predictions, axis=-1)
cm = confusion_matrix(labels, pred_labels)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix: DeBERTa-v3-base", fontsize=16)
plt.ylabel('Actual Category')
plt.xlabel('Predicted Category')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

print("\nDetailed Classification Report:")
print(classification_report(labels, pred_labels, target_names=labels))

In [ ]:
# 7. Save and Publish
model_path = "./final_resume_classifier"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model locally saved to {model_path}")

# To push directly to your Hugging Face Hub (ensure you ran `huggingface-cli login` in Colab):
# trainer.push_to_hub("BassemRamdan/resume-classifier-deberta")